# Course Tutor: fine-tuning the explanation style

This notebook LoRA fine-tunes a small open model (Qwen2.5-0.5B-Instruct) on ~90
instructor-style QA pairs from the course corpus, then compares the base model
vs. the tuned model on held-out questions using a structure rubric
(restate, explain, example, check-understanding question, on-topic).

**Runtime:** set it to a GPU. In Colab: Runtime > Change runtime type > T4 GPU.
Then Runtime > Run all. Takes about 15-20 minutes.

The result is a real base-vs-tuned score you can put in the README.


## 1. Install

In [ ]:
!pip install -q -U "transformers>=4.46" "trl>=0.12" "peft>=0.13" accelerate datasets

## 2. Load the training data from the repo

In [ ]:
import json, urllib.request

RAW_URL = "https://raw.githubusercontent.com/riya0920/course-tutor/main/backend/finetune/train.jsonl"
rows = [json.loads(l) for l in urllib.request.urlopen(RAW_URL).read().decode().splitlines() if l.strip()]
print("training examples:", len(rows))
print("sample target:\n", rows[0]["messages"][2]["content"][:200])

## 3. Load the base model

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"   # open, no gating, small and fast
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16, device_map="auto"
)
print("loaded", BASE_MODEL)

## 4. Held-out questions and a generation helper\n\nThese concepts are in the corpus but the questions are phrased differently from training.

In [ ]:
SYSTEM = ("You are a course tutor. Explain the concept in four steps: restate the "
          "question, explain the idea, give a concrete example, then ask one "
          "check-understanding question.")

HELDOUT = [
    ("Tell me about supervised learning.", "supervised learning"),
    ("What's the deal with overfitting?", "overfitting"),
    ("Break down gradient descent for me.", "gradient descent"),
    ("How would you describe backpropagation?", "backpropagation"),
    ("What should I know about regularization?", "regularization"),
    ("Explain the bias variance tradeoff.", "bias variance"),
    ("What is a support vector machine?", "support vector machine"),
    ("How does k-means clustering work?", "k-means clustering"),
    ("What does dropout do?", "dropout"),
    ("Tell me about principal component analysis.", "principal component analysis"),
]

def generate(m, question, max_new_tokens=220):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    enc = tokenizer.apply_chat_template(
        msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True
    )
    enc = {k: v.to(m.device) for k, v in enc.items()}
    out = m.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                     pad_token_id=tokenizer.eos_token_id)
    prompt_len = enc["input_ids"].shape[1]
    return tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True)

## 5. Baseline: generate with the untrained model

In [ ]:
base_outputs = [generate(model, q) for q, _ in HELDOUT]
print(base_outputs[0][:300])

## 6. Fine-tune with LoRA

In [ ]:
from datasets import Dataset
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

def to_text(ex):
    return {"text": tokenizer.apply_chat_template(ex["messages"], tokenize=False)}

ds = Dataset.from_list(rows).map(to_text, remove_columns=["messages"])

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

cfg = SFTConfig(
    output_dir="tuned", per_device_train_batch_size=2, gradient_accumulation_steps=4,
    num_train_epochs=3, learning_rate=2e-4, logging_steps=10,
    dataset_text_field="text", report_to="none", fp16=True,
)

trainer = SFTTrainer(model=model, train_dataset=ds, args=cfg, peft_config=lora)
trainer.train()

## 7. Tuned: generate with the fine-tuned model

In [ ]:
tuned_model = trainer.model
tuned_outputs = [generate(tuned_model, q) for q, _ in HELDOUT]
print(tuned_outputs[0][:300])

## 8. Score base vs. tuned on the 5-point structure rubric

One point each for: restating the concept, giving a real explanation, including
an example, ending with a check-understanding question, and staying on topic.

In [ ]:
def score(text, concept):
    t = text.lower()
    first = t.split(".")[0]
    tail = " ".join(text.strip().split()[-30:])
    s = 0
    s += 1 if any(w in first for w in concept.split()) else 0          # restate
    s += 1 if len(text.split()) >= 25 else 0                           # explanation
    s += 1 if any(k in t for k in ["for example", "for instance", "imagine",
                                    "consider", "e.g", "such as", "suppose"]) else 0  # example
    s += 1 if "?" in tail else 0                                       # check question
    s += 1 if any(w in t for w in concept.split()) else 0             # on-topic
    return s

base_scores = [score(o, c) for o, (_, c) in zip(base_outputs, HELDOUT)]
tuned_scores = [score(o, c) for o, (_, c) in zip(tuned_outputs, HELDOUT)]

print(f"{'question':<45}{'base':>6}{'tuned':>7}")
for (q, _), b, t in zip(HELDOUT, base_scores, tuned_scores):
    print(f"{q[:44]:<45}{b:>6}{t:>7}")

ba = sum(base_scores) / len(base_scores)
ta = sum(tuned_scores) / len(tuned_scores)
print("\n" + "=" * 58)
print(f"base  average: {ba:.2f} / 5")
print(f"tuned average: {ta:.2f} / 5")
print(f"improvement:   {ta - ba:+.2f}")

## 9. (Optional) LLM-as-judge

If you want an LLM judge instead of the rubric, set your Gemini key and run this.
It scores each explanation 1-5 with Gemini.

In [ ]:
# import os
# os.environ["GOOGLE_API_KEY"] = "your-key"
# !pip install -q openai
# from openai import OpenAI
# c = OpenAI(api_key=os.environ["GOOGLE_API_KEY"],
#            base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
# RUBRIC = ("Score this explanation 1-5 on whether it restates the question, explains "
#           "clearly, gives an example, and ends with a check question. Reply with only a digit.")
# def judge(text):
#     r = c.chat.completions.create(model="gemini-2.0-flash", max_tokens=4,
#         messages=[{"role":"system","content":RUBRIC},{"role":"user","content":text}])
#     ds = "".join(ch for ch in (r.choices[0].message.content or "") if ch.isdigit())
#     return int(ds[:1]) if ds else 3
# print("base :", sum(judge(o) for o in base_outputs)/len(base_outputs))
# print("tuned:", sum(judge(o) for o in tuned_outputs)/len(tuned_outputs))

## Done

Put the two averages into the README (the base vs. tuned line). To use the tuned
model in the app, save the adapter (`trainer.model.save_pretrained('adapter')`),
download it, and serve it behind the `/explain` endpoint's `tuned` option.